# EDA on cached listening sample

Follow-up to `01_eda.ipynb`. That notebook reported group medians; this one looks at **distributions** for findings where medians alone were suspicious or uninformative.

## Questions
1. **No-history group** — the 2,479 sampled customers with zero pre-cutoff listening records: what is their churn rate?
2. **Observation history span** — median was 587 (churn) vs 571 (non-churn). Is that a real shift or noise?
3. **Recency puzzle** — median "days since last positive log" was 1 for **both** groups. Where is the mass actually?
4. **Listening volume non-signal** — group medians for 28-day listening were nearly identical. Is there tail signal under the median?
5. **Recent/baseline ratio** — group medians were 0.83 vs 0.88. What is the shape of the distribution?

## Inputs
- `data/processed/eda/churn-eda/cohort_cache/precutoff_listening_sample.csv` (305 MB)
- `data/processed/eda/churn-eda/cohort_cache/precutoff_listening_manifest.json`
- `train_v2.csv` (loaded from the local Kaggle output cache; teammates would need their own copy)

## Scope
Analysis only — no feature engineering, no model. Figures saved under `data/processed/eda/02_cache/`. Outputs cleared before commit (`AGENTS.md:14`).

## 1. Setup

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
EDA_OUT = ROOT / "data" / "processed" / "eda" / "churn-eda"
CACHE_DIR = EDA_OUT / "cohort_cache"
FIG_DIR = ROOT / "data" / "processed" / "eda" / "02_cache"
FIG_DIR.mkdir(parents=True, exist_ok=True)

CUTOFF = pd.Timestamp("2017-03-01")
RECENT_START = CUTOFF - pd.Timedelta(days=28)
BASELINE_START = CUTOFF - pd.Timedelta(days=90)

assert CACHE_DIR.exists(), f"Missing cache at {CACHE_DIR}; see repo data README."
print("Cache:", CACHE_DIR)
print("Figures:", FIG_DIR)

## 2. Load cache and verify fingerprint

The manifest records the exact sample we're analysing. Printing the fingerprint makes it obvious if someone's working from a different cache version.

In [ ]:
manifest = json.loads((CACHE_DIR / "precutoff_listening_manifest.json").read_text())
fp = manifest["fingerprint"]
print(f"seed={fp['seed']}  cutoff={fp['cutoff']}  sample_size={fp['sample_size']:,}")
print(f"sample_ids_sha256={fp['sample_ids_sha256'][:16]}\u2026")
print(f"scanned_rows={manifest['scanned_rows']:,}  selected_rows={manifest['selected_rows']:,}")

listening = pd.read_csv(
    CACHE_DIR / "precutoff_listening_sample.csv",
    dtype={"msno": "string", "date": "string", "total_secs": "float64"},
)
listening["parsed_date"] = pd.to_datetime(listening["date"], format="%Y%m%d", errors="coerce")
listening = listening.dropna(subset=["parsed_date"])
print(f"\nListening rows: {len(listening):,}")
print(f"Customers with any records: {listening['msno'].nunique():,} / {fp['sample_size']:,}")

## 3. Join labels

The cache is intentionally label-free. We now join the supplied churn labels from `train_v2.csv` for one row per sampled customer, matching the same deterministic sample (seed 42) used in `01_eda.ipynb` cell 30.

In [ ]:
CANDIDATES = [
    EDA_OUT / "cache" / "train_v2" / "data" / "churn_comp_refresh" / "train_v2.csv",
    ROOT / "data" / "raw" / "train_v2.csv",
]
label_path = next((p for p in CANDIDATES if p.exists()), None)
assert label_path is not None, (
    "train_v2.csv not found locally. Put it at data/raw/train_v2.csv, "
    "or run 01_eda.ipynb on Kaggle to populate the cache under data/processed/eda/churn-eda/cache/."
)
print("Labels from:", label_path.relative_to(ROOT))

all_labels = pd.read_csv(label_path, dtype={"msno": "string", "is_churn": "Int8"})
cohort_ids = all_labels[["msno"]].sample(n=fp["sample_size"], random_state=fp["seed"]).sort_values("msno").reset_index(drop=True)
cohort = cohort_ids.merge(all_labels, on="msno", validate="one_to_one")
cohort["outcome"] = cohort["is_churn"].map({0: "Non-churn", 1: "Churn"})
print(f"Cohort rows: {len(cohort):,}; churn rate: {cohort['is_churn'].mean() * 100:.2f}%")

## 4. Question 1 — the no-history group

`01_eda.ipynb` found 2,479 sampled customers with zero pre-cutoff listening records. What churn rate do they have, compared with customers that do have records?

If the no-history group's churn rate differs sharply from the cohort baseline, that's useful: "missingness itself is signal".

In [ ]:
ids_with_records = set(listening["msno"].unique())
cohort["has_any_records"] = cohort["msno"].isin(ids_with_records)

no_history = cohort.groupby("has_any_records").agg(
    customers=("is_churn", "size"),
    churners=("is_churn", "sum"),
)
no_history["churn_percentage"] = 100 * no_history["churners"] / no_history["customers"]
no_history.index = no_history.index.map({True: "Has records", False: "No records at all"})
print("Churn rate by history availability")
display(no_history)

overall = 100 * cohort["is_churn"].mean()
nh_rate = no_history.loc["No records at all", "churn_percentage"]
print(f"\nCohort baseline: {overall:.2f}%")
print(f"No-history group: {nh_rate:.2f}%  (lift = {nh_rate / overall:.2f}x)")

## 5. Question 2 — observation history span

Compute each customer's observed history span (days from first valid listening record to the cutoff) and plot the distribution split by churn group. The medians were 587 vs 571 — a 16-day gap, which could reflect either differing tenure or a few older-signup outliers.

Only customers with at least one valid record contribute; the no-history group is excluded by construction.

In [ ]:
valid_mask = (
    listening["total_secs"].notna()
    & listening["total_secs"].ge(0)
    & ~listening["total_secs"].isin([np.inf, -np.inf])
)
valid_listening = listening.loc[valid_mask]
first_valid = valid_listening.groupby("msno")["parsed_date"].min()
obs_span = (CUTOFF - first_valid).dt.days.rename("obs_span_days")

cohort_span = cohort.set_index("msno").join(obs_span, how="inner")
print(cohort_span.groupby("outcome")["obs_span_days"].describe().round(1))

fig, ax = plt.subplots(figsize=(8, 4))
for outcome, color in [("Non-churn", "#4C72B0"), ("Churn", "#C44E52")]:
    values = cohort_span.loc[cohort_span["outcome"].eq(outcome), "obs_span_days"]
    ax.hist(values, bins=60, alpha=0.5, label=f"{outcome} (n={len(values):,})", color=color, density=True)
ax.set(xlabel="Observed history span (days)", ylabel="Density",
       title="Days from first valid listening record to 2017-03-01, by outcome")
ax.legend()
fig.tight_layout(); fig.savefig(FIG_DIR / "obs_span_distribution.png", dpi=150); plt.show()

## 6. Question 3 — recency puzzle

Both groups had **median = 1 day** since the last positive-duration log, which is suspicious. It would mean the typical churner listened yesterday. Plot the full distribution to see where the mass sits and whether there's a bimodal split (listened recently vs. gone silent) that the median hides.

In [ ]:
positive = valid_listening.loc[valid_listening["total_secs"] > 0]
last_positive = positive.groupby("msno")["parsed_date"].max()
days_since = (CUTOFF - last_positive).dt.days.rename("days_since_last_positive_log")

cohort_rec = cohort.set_index("msno").join(days_since, how="inner")
print(cohort_rec.groupby("outcome")["days_since_last_positive_log"].describe(
    percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95]
).round(1))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for outcome, color in [("Non-churn", "#4C72B0"), ("Churn", "#C44E52")]:
    v = cohort_rec.loc[cohort_rec["outcome"].eq(outcome), "days_since_last_positive_log"]
    axes[0].hist(v, bins=np.arange(0, 60, 1), alpha=0.5, label=outcome, color=color, density=True)
    axes[1].hist(v, bins=np.arange(0, 400, 10), alpha=0.5, label=outcome, color=color, density=True)
axes[0].set(xlabel="Days since last positive log (0\u201360)", ylabel="Density", title="Recent-recency detail")
axes[1].set(xlabel="Days since last positive log (0\u2013400)", ylabel="Density", title="Full range")
for ax in axes: ax.legend()
fig.suptitle("Days since last positive-duration log, by outcome")
fig.tight_layout(); fig.savefig(FIG_DIR / "recency_distribution.png", dpi=150); plt.show()

## 7. Question 4 — listening volume: is there tail signal?

28-day medians were 1,284 (churn) vs 1,223 (non-churn) minutes — nearly identical, so the median rules it out as useful by itself. But medians collapse the whole distribution to one point. If churners have a heavier left tail (lots of near-zero listeners) and non-churners don't, that's still signal — just not visible via the median.

In [ ]:
recent = valid_listening.loc[valid_listening["parsed_date"].ge(RECENT_START)]
minutes_28d = (recent.groupby("msno")["total_secs"].sum() / 60).rename("minutes_28d")
cohort_vol = cohort.set_index("msno").join(minutes_28d, how="left").fillna({"minutes_28d": 0.0})

print(cohort_vol.groupby("outcome")["minutes_28d"].describe(
    percentiles=[0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95]
).round(1))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
clipped = cohort_vol["minutes_28d"].clip(upper=5000)
for outcome, color in [("Non-churn", "#4C72B0"), ("Churn", "#C44E52")]:
    v = clipped.loc[cohort_vol["outcome"].eq(outcome)]
    axes[0].hist(v, bins=50, alpha=0.5, label=outcome, color=color, density=True)
    log_v = np.log10(cohort_vol.loc[cohort_vol["outcome"].eq(outcome), "minutes_28d"] + 1)
    axes[1].hist(log_v, bins=50, alpha=0.5, label=outcome, color=color, density=True)
axes[0].set(xlabel="Recorded minutes in 28d (clipped at 5,000)", ylabel="Density", title="Linear scale")
axes[1].set(xlabel="log10(minutes + 1)", ylabel="Density", title="Log scale (reveals tails)")
for ax in axes: ax.legend()
fig.suptitle("28-day recorded minutes, by outcome")
fig.tight_layout(); fig.savefig(FIG_DIR / "volume_distribution.png", dpi=150); plt.show()

## 8. Question 5 — recent/baseline ratio

Compute the ratio of recent 28-day minutes-per-day to the preceding 62-day minutes-per-day, mirroring `01_eda.ipynb` §16. Eligibility: observed at least 90 days before the cutoff AND no unusable records in that 90-day window AND baseline rate strictly positive. Group medians were 0.83 (churn) vs 0.88 (non-churn).

Plot the distribution of eligible customers' ratios by outcome. The vertical line at 1.0 marks "no change"; values below mean recorded listening dropped recently.

In [ ]:
unusable_mask = ~valid_mask | listening.duplicated(["msno", "parsed_date"], keep=False)
unusable_in_90d_ids = set(
    listening.loc[unusable_mask & listening["parsed_date"].ge(BASELINE_START), "msno"]
)
observed_before_90d = first_valid.le(BASELINE_START)

recent_min_per_day = (recent.groupby("msno")["total_secs"].sum() / 60 / 28)
baseline = valid_listening.loc[
    valid_listening["parsed_date"].ge(BASELINE_START) & valid_listening["parsed_date"].lt(RECENT_START)
]
baseline_min_per_day = (baseline.groupby("msno")["total_secs"].sum() / 60 / 62)

ratios = pd.DataFrame({
    "recent": recent_min_per_day,
    "baseline": baseline_min_per_day,
}).reindex(cohort["msno"]).fillna(0.0)
ratios["eligible"] = ratios.index.isin(observed_before_90d[observed_before_90d].index) & ~ratios.index.isin(unusable_in_90d_ids)
ratios["ratio"] = (ratios["recent"] / ratios["baseline"].replace(0, np.nan)).where(ratios["eligible"] & ratios["baseline"].gt(0))

cohort_ratio = cohort.set_index("msno").join(ratios, how="left")
defined = cohort_ratio.dropna(subset=["ratio"])
print("Eligible customers with defined ratio, by outcome")
print(defined.groupby("outcome")["ratio"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).round(2))

fig, ax = plt.subplots(figsize=(8, 4))
clipped = defined["ratio"].clip(upper=3)
for outcome, color in [("Non-churn", "#4C72B0"), ("Churn", "#C44E52")]:
    v = clipped.loc[defined["outcome"].eq(outcome)]
    ax.hist(v, bins=50, alpha=0.5, label=f"{outcome} (n={len(v):,})", color=color, density=True)
ax.axvline(1.0, color="grey", linestyle="--", linewidth=1)
ax.set(xlabel="Recent 28d minutes-per-day / earlier 62d minutes-per-day (clipped at 3)",
       ylabel="Density", title="Personal-baseline ratio, by outcome")
ax.legend()
fig.tight_layout(); fig.savefig(FIG_DIR / "ratio_distribution.png", dpi=150); plt.show()

## 9. Findings and open questions

Claims below describe associations in the fixed 20,000-customer sample (seed 42, cutoff 2017-03-01). They are not causal and not model performance.

### Headline

Listening-behaviour features alone contain weak and partially counter-intuitive churn signal in this sample. The strongest finding from `01_eda.ipynb` — payment behaviour (auto-renew OFF is ~9x churn vs auto-renew ON) — remains the real signal. This notebook's role is to show the shapes of distributions whose medians were uninformative; it does not uncover a hidden listening-only predictor.

### Per-question findings

- **Q1 — No-history group (inverted finding):** customers with zero pre-cutoff listening records churn at **5.65%** (140 / 2,479), *below* the cohort baseline of 8.78%. Customers with any records churn at **9.22%** (1,615 / 17,521). This is opposite to the natural expectation that silent customers are more likely to churn. Plausible explanations (none verified here): lifetime / bundled / gift subscriptions that don't require active listening; data-availability gaps where listening events weren't captured but transactions were; cohorts whose subscription predates the listening-log window. Needs cross-check against transactions and member profile before use.
- **Q2 — Observation history span:** distributions overlap almost completely with a massive density spike near ~780 days — the maximum reachable from the 2015-01-01 data start. The 16-day median gap from `01_eda.ipynb` is noise, not a real shift. Churners have a very slightly thicker left tail (shorter observed span), but not usefully so.
- **Q3 — Recency puzzle resolved:** both groups have ~57% of customers at exactly 1 day since the last positive log. This is a **selection artefact**: the comparison is restricted to customers with records, who are mostly regular users listening up to the cutoff. The distributions are visually indistinguishable through the full 400-day range. The median-of-1 is not a null finding worth reporting as a feature by itself.
- **Q4 — 28-day listening volume:** the log-scale panel reveals a clearly **bimodal** distribution — one spike at log=0 (zero recorded minutes) and a near-normal mass around log10 ~ 3 (~1,000 minutes). Counter to expectation, non-churn has a *larger* zero spike than churn in the linear panel. The main body overlaps. The 5,000-minute spike on the linear panel is a clipping artefact.
- **Q5 — Recent/baseline ratio:** the central mass overlaps, but the **tails diverge** — churn has visibly higher density at ratio ~ 0 and at ratio >= 3. This is the only listening feature in the notebook showing a non-overlapping shape by outcome. Even so, the effect is small; eligibility was 13,379 non-churn + 1,294 churn customers (82% of each outcome group).

### Implications for `03_features.ipynb`

- `has_any_records` (binary): keep as a candidate feature, but with the **inverted direction** — "no records" is lower churn risk in this sample, not higher. The feature's categorical nature (missingness as a signal) is still useful; the sign is just the opposite of the naive prior.
- `obs_span_days`: deprioritise as a continuous feature in this sample — distribution overlap is too complete. If used, note that it is recorded listening tenure, not customer tenure.
- `days_since_last_positive_log`: deprioritise standalone, keep as part of a wider recency panel. Must remain **missing** when there is no positive log — imputation would collapse an informative category.
- `minutes_28d`: raw value adds little; `log10(minutes_28d + 1)` captures the bimodality in a form linear models can use. Pair with an explicit `has_recent_listening` binary derived from the zero spike.
- `recent_to_baseline_ratio`: keep, with the eligibility flag kept separate (no imputation). Consider derived binaries like `ratio_lt_0_1` and `ratio_gt_2` to capture the tail-density difference rather than relying on the overlapping centre.
- Overall: a model built on listening features alone will struggle in this sample. The feature pipeline should include the payment behaviour (`latest_auto_renew_status`, `cancellation_events_28d`, `days_since_last_transaction`) from `01_eda.ipynb` §17.

### Open questions to resolve before features

- Who are the 2,479 no-history customers in terms of their *transactions*? Are they predominantly auto-renew=1 lifetime-style plans? This decides whether `has_any_records` is a lifestyle proxy or a data-availability proxy.
- Reconfirm that the deterministic resample in cell 6 produces the same 20,000 IDs as `01_eda.ipynb` cell 30. The manifest `sample_ids_sha256` should match: `14d1b0ae...e41eb1`.
- Many open items from `DECISIONS.md:47-55` remain — label construction, eligibility, cohort maturity, metrics. This notebook does not advance any of them.